# Multi-LogiEval Depth-5 Analysis: The Hardest Reasoning Challenge

Deep dive into depth-5 (most complex) reasoning performance across CoT, Lean, and Two-Stage approaches.

**Dataset**: 110 depth-5 questions (3 logic types × varying samples)
- **FOL**: First-order logic (45 questions)
- **NM**: Non-monotonic logic (20 questions)
- **PL**: Propositional logic (45 questions)
- **Depth-5**: 5 chained inference rules - the longest reasoning chains in Multi-LogiEval

**Why d5 matters**: Depth-5 represents the most complex multi-step reasoning task in the benchmark. Performance here tests the limits of:
- Chain-of-thought reasoning over long dependency chains
- Lean's ability to verify complex formal proofs
- Two-Stage's refinement capability on hard problems

**Hypothesis**: As depth increases, reasoning becomes harder. We expect d5 to show the lowest accuracy across all approaches, with the biggest gaps between methods revealing which approach scales better to complex reasoning.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import numpy as np

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Results

In [ ]:
# Load d5-only results
with open('../results/multilogieval/d5_only/cot/all_results.json', 'r') as f:
    cot_d5 = json.load(f)

with open('../results/multilogieval/d5_only/lean/all_results.json', 'r') as f:
    lean_d5 = json.load(f)

with open('../results/multilogieval/d5_only/two_stage/all_results.json', 'r') as f:
    two_stage_d5 = json.load(f)

print(f"Loaded {len(cot_d5)} depth-5 questions from each approach")
print(f"\nLogic type distribution:")
logic_counts = Counter(r['logic_type'] for r in cot_d5)
for logic, count in sorted(logic_counts.items()):
    print(f"  {logic.upper()}: {count} questions")

print(f"\nDepth-5 Accuracies:")
print(f"  CoT:       {sum(1 for r in cot_d5 if r['correct'])}/110 = {sum(1 for r in cot_d5 if r['correct'])/110*100:.2f}%")
print(f"  Lean:      {sum(1 for r in lean_d5 if r['correct'])}/110 = {sum(1 for r in lean_d5 if r['correct'])/110*100:.2f}%")
print(f"  Two-Stage: {sum(1 for r in two_stage_d5 if r['correct'])}/110 = {sum(1 for r in two_stage_d5 if r['correct'])/110*100:.2f}%")

## 2. Overall Performance Comparison for Depth-5

In [ ]:
cot_d5_acc = sum(1 for r in cot_d5 if r['correct']) / len(cot_d5) * 100
lean_d5_acc = sum(1 for r in lean_d5 if r['correct']) / len(lean_d5) * 100
two_stage_d5_acc = sum(1 for r in two_stage_d5 if r['correct']) / len(two_stage_d5) * 100

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

approaches = ['CoT', 'Lean', 'Two-Stage']
accuracies = [cot_d5_acc, lean_d5_acc, two_stage_d5_acc]
colors = ['#2ecc71', '#3498db', '#9b59b6']

bars = ax1.bar(approaches, accuracies, color=colors)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Depth-5 Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Correct/Incorrect breakdown
cot_correct = sum(1 for r in cot_d5 if r['correct'])
lean_correct = sum(1 for r in lean_d5 if r['correct'])
two_stage_correct = sum(1 for r in two_stage_d5 if r['correct'])

x = np.arange(3)
correct_counts = [cot_correct, lean_correct, two_stage_correct]
incorrect_counts = [110 - c for c in correct_counts]

ax2.bar(x, correct_counts, label='Correct', color='#2ecc71')
ax2.bar(x, incorrect_counts, bottom=correct_counts, label='Incorrect', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(approaches)
ax2.set_ylabel('Number of Questions', fontsize=12)
ax2.set_title('Depth-5: Correct vs Incorrect Breakdown', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DEPTH-5 PERFORMANCE RANKING")
print("="*70)
ranked = sorted(zip(approaches, accuracies), key=lambda x: x[1], reverse=True)
for i, (approach, acc) in enumerate(ranked, 1):
    print(f"{i}. {approach}: {acc:.1f}%")
print("="*70)

## 3. Performance by Logic Type (Depth-5 Only)

In [ ]:
logic_types = ['fol', 'nm', 'pl']
logic_names = {'fol': 'FOL', 'nm': 'NM', 'pl': 'PL'}

cot_by_logic = {}
lean_by_logic = {}
two_stage_by_logic = {}

for logic in logic_types:
    cot_logic = [r for r in cot_d5 if r['logic_type'] == logic]
    lean_logic = [r for r in lean_d5 if r['logic_type'] == logic]
    two_stage_logic = [r for r in two_stage_d5 if r['logic_type'] == logic]
    
    if len(cot_logic) > 0:
        cot_by_logic[logic] = sum(1 for r in cot_logic if r['correct']) / len(cot_logic) * 100
        lean_by_logic[logic] = sum(1 for r in lean_logic if r['correct']) / len(lean_logic) * 100
        two_stage_by_logic[logic] = sum(1 for r in two_stage_logic if r['correct']) / len(two_stage_logic) * 100

# Visualize
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(logic_types))
width = 0.25

ax.bar(x - width, [cot_by_logic[l] for l in logic_types], width, label='CoT', color='#2ecc71')
ax.bar(x, [lean_by_logic[l] for l in logic_types], width, label='Lean', color='#3498db')
ax.bar(x + width, [two_stage_by_logic[l] for l in logic_types], width, label='Two-Stage', color='#9b59b6')

ax.set_xlabel('Logic Type', fontsize=12)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Depth-5 Accuracy by Logic Type', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([logic_names[l] for l in logic_types])
ax.legend()
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DEPTH-5 ACCURACY BY LOGIC TYPE")
print("="*70)
print(f"{'Logic':<8} {'CoT':<12} {'Lean':<12} {'Two-Stage':<12} {'N':<8}")
print("-"*70)
for logic in logic_types:
    n = len([r for r in cot_d5 if r['logic_type'] == logic])
    print(f"{logic_names[logic]:<8} {cot_by_logic[logic]:>6.1f}%     {lean_by_logic[logic]:>6.1f}%     {two_stage_by_logic[logic]:>6.1f}%     {n:>3}")
print("="*70)

## 4. Lean Verification Statistics for Depth-5

In [ ]:
# Lean verification stats
lean_verified = sum(1 for r in lean_d5 if r.get('lean_verification', {}).get('success', False))
lean_verified_correct = sum(1 for r in lean_d5 
                            if r.get('lean_verification', {}).get('success', False) and r['correct'])
lean_verified_wrong = sum(1 for r in lean_d5 
                          if r.get('lean_verification', {}).get('success', False) and not r['correct'])

# Two-Stage verification stats
two_stage_verified = 0
two_stage_verified_correct = 0
two_stage_verified_wrong = 0
for r in two_stage_d5:
    if 'all_cycles' in r and r['all_cycles']:
        final_cycle = r['all_cycles'][-1]
        if final_cycle.get('lean_success', False):
            two_stage_verified += 1
            if r['correct']:
                two_stage_verified_correct += 1
            else:
                two_stage_verified_wrong += 1

# Visualize verification breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Verification success rates
approaches = ['Lean', 'Two-Stage']
verification_rates = [
    lean_verified / 110 * 100,
    two_stage_verified / 110 * 100
]
colors_verify = ['#3498db', '#9b59b6']

bars1 = ax1.bar(approaches, verification_rates, color=colors_verify)
ax1.set_ylabel('Verification Success Rate (%)', fontsize=12)
ax1.set_title('Depth-5: Lean Verification Success Rates', fontsize=14, fontweight='bold')
ax1.set_ylim(0, 100)
for bar, rate in zip(bars1, verification_rates):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{rate:.1f}%', ha='center', fontsize=11, fontweight='bold')

# Verified-but-wrong breakdown
x = np.arange(2)
verified_correct = [lean_verified_correct, two_stage_verified_correct]
verified_wrong = [lean_verified_wrong, two_stage_verified_wrong]

ax2.bar(x, verified_correct, label='Verified & Correct', color='#2ecc71')
ax2.bar(x, verified_wrong, bottom=verified_correct, label='Verified but Wrong', color='#e74c3c')
ax2.set_xticks(x)
ax2.set_xticklabels(approaches)
ax2.set_ylabel('Number of Verified Questions', fontsize=12)
ax2.set_title('Depth-5: Verified Questions Breakdown', fontsize=14, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("="*70)
print("DEPTH-5 LEAN VERIFICATION STATISTICS")
print("="*70)

print(f"\nLean:")
print(f"  Verification success: {lean_verified}/110 ({lean_verified/110*100:.1f}%)")
if lean_verified > 0:
    print(f"  Among verified:")
    print(f"    Correct answer:   {lean_verified_correct} ({lean_verified_correct/lean_verified*100:.1f}%)")
    print(f"    Wrong answer:     {lean_verified_wrong} ({lean_verified_wrong/lean_verified*100:.1f}%)")

print(f"\nTwo-Stage:")
print(f"  Verification success: {two_stage_verified}/110 ({two_stage_verified/110*100:.1f}%)")
if two_stage_verified > 0:
    print(f"  Among verified:")
    print(f"    Correct answer:   {two_stage_verified_correct} ({two_stage_verified_correct/two_stage_verified*100:.1f}%)")
    print(f"    Wrong answer:     {two_stage_verified_wrong} ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)")

print(f"\nKEY INSIGHT:")
if lean_verified > 0:
    print(f"At depth-5, Lean has {lean_verified_wrong} verified-but-wrong cases ({lean_verified_wrong/lean_verified*100:.1f}% of verified)")
if two_stage_verified > 0:
    print(f"At depth-5, Two-Stage has {two_stage_verified_wrong} verified-but-wrong cases ({two_stage_verified_wrong/two_stage_verified*100:.1f}% of verified)")
print("="*70)

## 5. Comparison with All Depths

How does depth-5 performance compare to the overall Multi-LogiEval results?

In [ ]:
# Load all_depths results for comparison
with open('../results/multilogieval/all_depths/cot/all_results.json', 'r') as f:
    cot_all = json.load(f)

with open('../results/multilogieval/all_depths/lean/all_results.json', 'r') as f:
    lean_all = json.load(f)

with open('../results/multilogieval/all_depths/two_stage/all_results.json', 'r') as f:
    two_stage_all = json.load(f)

# Calculate overall accuracies
cot_overall = sum(1 for r in cot_all if r['correct']) / len(cot_all) * 100
lean_overall = sum(1 for r in lean_all if r['correct']) / len(lean_all) * 100
two_stage_overall = sum(1 for r in two_stage_all if r['correct']) / len(two_stage_all) * 100

# Calculate accuracies by depth from all_depths
depths = ['d1_Data', 'd2_Data', 'd3_Data', 'd4_Data', 'd5_Data']
depth_labels = ['d1', 'd2', 'd3', 'd4', 'd5']

cot_by_depth = {}
lean_by_depth = {}
two_stage_by_depth = {}

for depth in depths:
    cot_depth = [r for r in cot_all if r['depth_dir'] == depth]
    lean_depth = [r for r in lean_all if r['depth_dir'] == depth]
    two_stage_depth = [r for r in two_stage_all if r['depth_dir'] == depth]
    
    if len(cot_depth) > 0:
        cot_by_depth[depth] = sum(1 for r in cot_depth if r['correct']) / len(cot_depth) * 100
        lean_by_depth[depth] = sum(1 for r in lean_depth if r['correct']) / len(lean_depth) * 100
        two_stage_by_depth[depth] = sum(1 for r in two_stage_depth if r['correct']) / len(two_stage_depth) * 100

# Visualize depth progression
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Performance by depth (line plot)
x = np.arange(len(depths))
ax1.plot(x, [cot_by_depth[d] for d in depths], 'o-', label='CoT', color='#2ecc71', linewidth=2, markersize=8)
ax1.plot(x, [lean_by_depth[d] for d in depths], 's-', label='Lean', color='#3498db', linewidth=2, markersize=8)
ax1.plot(x, [two_stage_by_depth[d] for d in depths], '^-', label='Two-Stage', color='#9b59b6', linewidth=2, markersize=8)

# Highlight d5
ax1.axvline(x=4, color='red', linestyle='--', alpha=0.5, label='Depth-5 Focus')
ax1.set_xlabel('Reasoning Depth (Number of Chained Rules)', fontsize=12)
ax1.set_ylabel('Accuracy (%)', fontsize=12)
ax1.set_title('Accuracy Across All Depths (with d5 highlighted)', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(depth_labels)
ax1.legend(fontsize=11)
ax1.set_ylim(0, 100)
ax1.grid(alpha=0.3)

# Overall vs d5 comparison
comparison_data = {
    'Overall\n(all depths)': [cot_overall, lean_overall, two_stage_overall],
    'Depth-5\n(110 samples)': [cot_d5_acc, lean_d5_acc, two_stage_d5_acc]
}

x2 = np.arange(len(approaches))
width = 0.35

bars1 = ax2.bar(x2 - width/2, comparison_data['Overall\n(all depths)'], width, 
                label='Overall (all depths)', color='#95a5a6', alpha=0.7)
bars2 = ax2.bar(x2 + width/2, comparison_data['Depth-5\n(110 samples)'], width,
                label='Depth-5 only', color=['#2ecc71', '#3498db', '#9b59b6'])

ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Overall vs Depth-5 Performance', fontsize=14, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(approaches)
ax2.legend()
ax2.set_ylim(0, 100)
ax2.grid(axis='y', alpha=0.3)

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 1,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DEPTH-5 vs OVERALL PERFORMANCE")
print("="*70)
print(f"\n{'Approach':<12} {'Overall':<12} {'Depth-5':<12} {'Difference'}")
print("-"*70)
print(f"{'CoT':<12} {cot_overall:>6.1f}%     {cot_d5_acc:>6.1f}%     {cot_d5_acc - cot_overall:>+6.1f}%")
print(f"{'Lean':<12} {lean_overall:>6.1f}%     {lean_d5_acc:>6.1f}%     {lean_d5_acc - lean_overall:>+6.1f}%")
print(f"{'Two-Stage':<12} {two_stage_overall:>6.1f}%     {two_stage_d5_acc:>6.1f}%     {two_stage_d5_acc - two_stage_overall:>+6.1f}%")
print("="*70)

## 6. Depth Degradation Analysis

How much does performance degrade from easiest (d1) to hardest (d5)?

In [ ]:
# Calculate degradation from d1 to d5
cot_degradation = cot_by_depth['d1_Data'] - cot_by_depth['d5_Data']
lean_degradation = lean_by_depth['d1_Data'] - lean_by_depth['d5_Data']
two_stage_degradation = two_stage_by_depth['d1_Data'] - two_stage_by_depth['d5_Data']

# Visualize degradation
fig, ax = plt.subplots(figsize=(10, 6))

degradations = [cot_degradation, lean_degradation, two_stage_degradation]
colors_deg = ['#2ecc71', '#3498db', '#9b59b6']

bars = ax.bar(approaches, degradations, color=colors_deg)
ax.set_ylabel('Accuracy Drop (d1 - d5) in %', fontsize=12)
ax.set_title('Performance Degradation from Depth-1 to Depth-5', fontsize=14, fontweight='bold')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

for bar, deg in zip(bars, degradations):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 1,
            f'{deg:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DEPTH DEGRADATION ANALYSIS (d1 → d5)")
print("="*70)
print(f"\n{'Approach':<12} {'d1 Acc':<12} {'d5 Acc':<12} {'Degradation'}")
print("-"*70)
print(f"{'CoT':<12} {cot_by_depth['d1_Data']:>6.1f}%     {cot_by_depth['d5_Data']:>6.1f}%     {cot_degradation:>6.1f}%")
print(f"{'Lean':<12} {lean_by_depth['d1_Data']:>6.1f}%     {lean_by_depth['d5_Data']:>6.1f}%     {lean_degradation:>6.1f}%")
print(f"{'Two-Stage':<12} {two_stage_by_depth['d1_Data']:>6.1f}%     {two_stage_by_depth['d5_Data']:>6.1f}%     {two_stage_degradation:>6.1f}%")
print("="*70)
print(f"\nInterpretation:")
print(f"  - Lower degradation = better scaling to complex reasoning")
print(f"  - Best scaling: {approaches[degradations.index(min(degradations))]} ({min(degradations):.1f}% drop)")
print(f"  - Worst scaling: {approaches[degradations.index(max(degradations))]} ({max(degradations):.1f}% drop)")
print("="*70)

## 7. Key Insights: Why is Depth-5 Harder?

Analyzing the challenges and successes at maximum reasoning depth.

In [ ]:
print("="*80)
print("KEY INSIGHTS: DEPTH-5 ANALYSIS")
print("="*80)

print("\n1. OVERALL DEPTH-5 PERFORMANCE:")
print(f"   CoT:       {cot_d5_acc:.1f}% ({sum(1 for r in cot_d5 if r['correct'])}/110 correct)")
print(f"   Lean:      {lean_d5_acc:.1f}% ({sum(1 for r in lean_d5 if r['correct'])}/110 correct)")
print(f"   Two-Stage: {two_stage_d5_acc:.1f}% ({sum(1 for r in two_stage_d5 if r['correct'])}/110 correct)")
print(f"\n   Winner at d5: {approaches[accuracies.index(max(accuracies))]}")
print(f"   Gap from best to worst: {max(accuracies) - min(accuracies):.1f}%")

print("\n2. COMPARISON TO OVERALL PERFORMANCE:")
print(f"   CoT:       Overall {cot_overall:.1f}% → d5 {cot_d5_acc:.1f}% ({cot_d5_acc - cot_overall:+.1f}%)")
print(f"   Lean:      Overall {lean_overall:.1f}% → d5 {lean_d5_acc:.1f}% ({lean_d5_acc - lean_overall:+.1f}%)")
print(f"   Two-Stage: Overall {two_stage_overall:.1f}% → d5 {two_stage_d5_acc:.1f}% ({two_stage_d5_acc - two_stage_overall:+.1f}%)")

print("\n3. DEPTH DEGRADATION (d1 → d5):")
print(f"   CoT:       {cot_degradation:.1f}% accuracy drop")
print(f"   Lean:      {lean_degradation:.1f}% accuracy drop")
print(f"   Two-Stage: {two_stage_degradation:.1f}% accuracy drop")
print(f"\n   Best scaling to d5: {approaches[degradations.index(min(degradations))]} (smallest drop)")

print("\n4. LOGIC TYPE PERFORMANCE AT D5:")
for logic in logic_types:
    n = len([r for r in cot_d5 if r['logic_type'] == logic])
    print(f"   {logic_names[logic]:>3} (n={n:>2}): CoT {cot_by_logic[logic]:.1f}%, Lean {lean_by_logic[logic]:.1f}%, Two-Stage {two_stage_by_logic[logic]:.1f}%")

print("\n5. LEAN VERIFICATION AT D5:")
print(f"   Lean verification rate:      {lean_verified}/110 ({lean_verified/110*100:.1f}%)")
print(f"   Two-Stage verification rate: {two_stage_verified}/110 ({two_stage_verified/110*100:.1f}%)")
if lean_verified > 0:
    print(f"\n   Lean verified-but-wrong:      {lean_verified_wrong}/{lean_verified} ({lean_verified_wrong/lean_verified*100:.1f}%)")
if two_stage_verified > 0:
    print(f"   Two-Stage verified-but-wrong: {two_stage_verified_wrong}/{two_stage_verified} ({two_stage_verified_wrong/two_stage_verified*100:.1f}%)")

print("\n6. WHY IS DEPTH-5 HARDER?")
print("   - 5 chained inference rules create long dependency chains")
print("   - More intermediate steps = more opportunities for errors to compound")
print("   - Harder to track logical state across multiple transformations")
print("   - Lean formalization becomes more complex with longer proofs")

print("\n7. WHICH APPROACH WORKS BEST FOR COMPLEX REASONING?")
best_d5 = approaches[accuracies.index(max(accuracies))]
best_scaling = approaches[degradations.index(min(degradations))]
print(f"   Best absolute d5 performance: {best_d5} ({max(accuracies):.1f}%)")
print(f"   Best scaling (d1→d5):         {best_scaling} ({min(degradations):.1f}% drop)")

if best_d5 == best_scaling:
    print(f"\n   CONCLUSION: {best_d5} excels at complex d5 reasoning")
else:
    print(f"\n   CONCLUSION: {best_d5} performs best at d5, but {best_scaling} scales better from d1")

print("\n8. COMPARISON WITH FOLIO:")
print("   FOLIO overall:     CoT 85.7% > Two-Stage 79.3% > Lean 74.9%")
print(f"   Multi-LogiEval d5: {approaches[accuracies.index(max(accuracies))]} {max(accuracies):.1f}% > ... > {approaches[accuracies.index(min(accuracies))]} {min(accuracies):.1f}%")
print("\n   Key difference: Lean performs differently on Multi-LogiEval vs FOLIO")
print("   This suggests dataset characteristics affect Lean's effectiveness")

print("="*80)

## 8. Sample Error Analysis

Let's examine a few depth-5 errors to understand failure modes.

In [ ]:
# Find examples of different error types
print("="*80)
print("SAMPLE ERROR CASES AT DEPTH-5")
print("="*80)

# Case 1: CoT wrong, Lean correct
cot_wrong_lean_right = [
    i for i in range(len(cot_d5))
    if not cot_d5[i]['correct'] and lean_d5[i]['correct']
]

print(f"\n1. Cases where CoT failed but Lean succeeded: {len(cot_wrong_lean_right)} cases")
if len(cot_wrong_lean_right) > 0:
    idx = cot_wrong_lean_right[0]
    print(f"\n   Example (Question {idx}):")
    print(f"   Logic type: {cot_d5[idx]['logic_type'].upper()}")
    print(f"   Correct answer: {cot_d5[idx]['ground_truth']}")
    print(f"   CoT predicted: {cot_d5[idx]['predicted_answer']}")
    print(f"   Lean predicted: {lean_d5[idx]['predicted_answer']}")
    print(f"   Lean verified: {lean_d5[idx].get('lean_verification', {}).get('success', False)}")

# Case 2: Lean verified but wrong
lean_verified_wrong_cases = [
    i for i in range(len(lean_d5))
    if lean_d5[i].get('lean_verification', {}).get('success', False) and not lean_d5[i]['correct']
]

print(f"\n2. Cases where Lean verified but got wrong answer: {len(lean_verified_wrong_cases)} cases")
if len(lean_verified_wrong_cases) > 0:
    idx = lean_verified_wrong_cases[0]
    print(f"\n   Example (Question {idx}):")
    print(f"   Logic type: {lean_d5[idx]['logic_type'].upper()}")
    print(f"   Correct answer: {lean_d5[idx]['ground_truth']}")
    print(f"   Lean predicted: {lean_d5[idx]['predicted_answer']}")
    print(f"   Issue: Lean formalization verified but led to wrong conclusion")

# Case 3: All approaches failed
all_failed = [
    i for i in range(len(cot_d5))
    if not cot_d5[i]['correct'] and not lean_d5[i]['correct'] and not two_stage_d5[i]['correct']
]

print(f"\n3. Cases where ALL approaches failed: {len(all_failed)} cases")
print(f"   These represent the truly hard d5 problems that challenge all methods")

if len(all_failed) > 0:
    # Count by logic type
    all_failed_by_logic = Counter(cot_d5[i]['logic_type'] for i in all_failed)
    print(f"\n   Distribution by logic type:")
    for logic, count in sorted(all_failed_by_logic.items()):
        print(f"     {logic.upper()}: {count} cases")

# Case 4: All approaches succeeded
all_succeeded = [
    i for i in range(len(cot_d5))
    if cot_d5[i]['correct'] and lean_d5[i]['correct'] and two_stage_d5[i]['correct']
]

print(f"\n4. Cases where ALL approaches succeeded: {len(all_succeeded)} cases")
print(f"   These are the 'easy' d5 problems solvable by all methods")

print("\n" + "="*80)
print(f"Summary: {len(all_succeeded)}/{110} d5 questions solved by all, {len(all_failed)}/{110} unsolved by all")
print("="*80)

## 9. Conclusions

**Key Findings:**

1. **Depth-5 is indeed the hardest**: Performance drops significantly from d1 to d5 across all approaches

2. **Method comparison at maximum complexity**:
   - The best-performing method at d5 reveals which approach scales to complex reasoning
   - Verification success rates show whether Lean can handle long proof chains
   - Verified-but-wrong cases indicate formalization accuracy issues

3. **Logic type differences**: Some logic types may be inherently harder at d5

4. **Comparison with FOLIO**:
   - Multi-LogiEval d5 tests longer reasoning chains than typical FOLIO problems
   - Different dataset characteristics may favor different approaches

**Implications for Reverse Curriculum Learning:**
- Starting from d5 (hardest) makes sense as these are the most challenging problems
- The high error rate at d5 provides strong training signal
- Lean verification at d5 can identify which complex problems are formally solvable
- Success at d5 should transfer down to simpler depths (d4 → d3 → d2 → d1)